# 01 · Seeing site-level heterogeneity
**Workshop: Beyond Interoperability — Federated ML for Research Data Curation**

Interoperability lets data cross institutional lines. It does *not* guarantee that a variable *means* the same thing at every site. Before we integrate any data or models, we quantify how the four hospitals in the UCI Heart Disease federation differ, and we separate two very different phenomena:

* **True distributional shift** — the populations genuinely differ (e.g. disease prevalence).
* **Measurement-induced heterogeneity** — the *measurement process* differs (e.g. a site never recorded serum cholesterol).


In [ ]:
# --- workshop bootstrap: make the package importable ---
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import numpy as np, pandas as pd
from fedconformal import data, conformal, evaluate as ev, federated, heterogeneity as het, viz, eda
viz.set_style()
%matplotlib inline
# load the federation once (each notebook is self-contained)
df = data.load_raw()
sites = data.load_sites(shared_scaler=True)

## The four sites
Cleveland, Hungary, Switzerland (Zurich/Basel) and the V.A. Long Beach each ran the *same* case-report form. That is what makes this dataset a natural federation.

In [ ]:
df = data.load_raw()
sites = data.load_sites(shared_scaler=True)
summary = data.summarize_sites(sites, df)
summary

In [ ]:
viz.plot_site_overview(summary);

Notice prevalence swings from **36% (Hungary)** to **94% (Switzerland)** — a large *true* label shift driven by referral patterns (Switzerland is a tertiary cardiac centre).

## Measurement-induced heterogeneity
A cholesterol of exactly 0 mg/dl is not a measurement — it is an *unrecorded* value. Watch which sites simply did not record certain labs.

In [ ]:
miss = het.missingness_report(df)
display(miss.style.format('{:.0%}'))
viz.plot_missingness(miss);

**Switzerland never recorded cholesterol (100% unrecorded).** If you naively pooled the raw feature, Switzerland's 'cholesterol = 0' would look like a population with impossibly low cholesterol — a curation error, not biology.

In [ ]:
viz.plot_feature_distributions(df, 'chol');

## A single-number covariate-shift alarm
Train a classifier to guess *which site* a patient came from. If it cannot do better than chance (AUC ≈ 0.5) the sites are exchangeable; AUC ≈ 1.0 signals strong covariate shift.

In [ ]:
auc = het.domain_auc_matrix(sites)
display(auc.round(2))
viz.plot_divergence_matrix(auc, 'Domain-classifier AUC (site vs site)', cmap='Oranges');

Every off-diagonal AUC is **0.93–1.00**: a model can almost perfectly tell any two sites apart. This is exactly the setting where a model — or a calibration — from one site is at risk of failing silently at another. That is what the next notebooks make precise.

### Exercise
1. Swap `shared_scaler=True` for `False` in `load_sites` and re-plot. What does per-site standardization *hide*?
2. Compute `het.js_divergence_matrix(df, 'age')` and compare with `'thalach'`. Which feature is most shifted across sites?